# Satellite nowcasting: U-Net vs persistence vs optical flow (Kaggle, GPU)

**Settings:** Internet **on**, accelerator **GPU T4**. Add the outputs of every ingest chunk as input data
(**Add Input → Your Work → notebook output**, once per chunk). The silver frames of all chunks are merged
into one gold Zarr here. Run with **Save & Run All**; results go to `/kaggle/working/runs`.

In [ ]:
!git clone -q --depth 1 https://github.com/ngocquangbiii06/hcmc-storm-nowcasting.git /tmp/stormcast
!pip install -q -e "/tmp/stormcast[ml]"
import os, sys, importlib
os.chdir('/tmp/stormcast'); sys.path.insert(0, '/tmp/stormcast/src')
# fail loudly here rather than 141 times per day later
for mod in ['satpy', 'pyresample', 'xarray', 'zarr', 'boto3'] + ['torch', 'cv2']:
    importlib.import_module(mod)
print(sys.version.split()[0], 'dependencies ok')

In [ ]:
import glob, yaml
from pathlib import Path
import pandas as pd
from stormcast.build_dataset import build_sat_zarr
from stormcast.config import load_config

# every ingest chunk has its own data root; make its catalog paths absolute and merge them
frames = []
for cat in sorted(glob.glob('/kaggle/input/**/catalog/himawari_frames', recursive=True)):
    root = Path(cat).parents[1]
    for part in sorted(Path(cat).glob('date=*.parquet')):
        df = pd.read_parquet(part)
        df['silver_path'] = [str(root / p) for p in df['silver_path']]
        frames.append(df)
frames = pd.concat(frames, ignore_index=True).drop_duplicates('nominal_time', keep='last')
frames = frames[frames['status'] == 'ok']
print('ok frames:', len(frames), '| days:', frames['nominal_time'].dt.date.nunique())

cfg = yaml.safe_load(open('configs/default.yaml'))
zarr = '/kaggle/working/data/gold/himawari_aug2025.zarr'
cfg['paths']['data_root'] = '/kaggle/working/data'
cfg['ml'].update(sat_zarr_path=zarr, output_dir='/kaggle/working/runs', num_workers=2)
yaml.safe_dump(cfg, open('configs/kaggle.yaml', 'w'))
print(build_sat_zarr(load_config('configs/kaggle.yaml'), frames, zarr))

In [ ]:
!python -m stormcast.ml.train_nowcast --config configs/kaggle.yaml --epochs 1 --max-train-batches 10

In [ ]:
!python -m stormcast.ml.train_nowcast --config configs/kaggle.yaml

In [ ]:
import json
from IPython.display import Image, display
run = sorted(glob.glob('/kaggle/working/runs/nowcast-*'))[-1]
display(Image(f'{run}/skill_vs_lead.png'))
test = json.load(open(f'{run}/metrics.json'))['test']
for method, per_lead in test.items():
    print(f'{method:13s}', '  '.join(f"{k}: {v['csi@235K']:.3f}" for k, v in per_lead.items()))